# Database check — what the warehouse holds right now

A scratch notebook for quick questions about the current state of the database. Edit the SQL and rerun.

**Before running:** open the warehouse tunnel in a terminal and leave it running:

    cd /Users/marcalexander/projects/ai_orchestrator_claude/ncaa_football/wt-c && scripts/warehouse_tunnel.sh

Queries are read-only: `cf.q()` refuses anything but a single `select` / `with`.

In [ ]:
import cfdb_conn as cf

# Which database, and how fresh is it? "warehouse" = what dbt has built (the default).
cf.check()

## How many Week 5 and Week 6 games carry a model forecast?

`predicted_margin` is the model's line; a game without one shows up as `games` > `with_forecast`.

In [ ]:
cf.q("""
    select week,
           count(*)                 as games,
           count(predicted_margin)  as with_forecast
    from serving.srv_game
    where season = 2026
      and week in (5, 6)
    group by 1
    order by 1
""", limit=None)

## This week's biggest disagreements with Vegas — possible betting opportunities

Every **FBS-vs-FBS** game this week that has **not kicked off yet** and carries both a Vegas line and our
model's line. "This week" = the week of the earliest FBS game not yet played.

| column | meaning |
|---|---|
| **Vegas line** | the market's spread from the home team's side (negative = home favored), e.g. `UGA -24` |
| **My line** | our model's spread, same form |
| **Gap** | Vegas line − my line. **<span style='color:#0b3d91'>Blue +</span>: you like the home team more than Vegas. <span style='color:#8a3b00'>Orange −</span>: you like the away team more** |
| **Matchup** | link to that game's Matchup page on the site |

**Sorted by Gap, ascending:** the biggest leans toward the **away** team are at the top, the biggest toward the
**home** team at the bottom, and the games where you and Vegas agree sit in the middle. Change `MIN_GAP` to show
only the big ones.

⚠️ A gap is a disagreement, not proof of value: in the 2025 backtest, games where the model differed from Vegas by
6+ points covered only 13–20–1. Lines also move until kickoff, so rerun close to game time.

In [ ]:
import pandas as pd
from IPython.display import HTML, Markdown, display

SITE = "https://cfdb.marc4data.com"
MIN_GAP = 0.0          # show games whose |gap| is at least this many points (0 = every game)

g = cf.q("""
    select game_id, season, week, start_date, is_completed,
           away_team, home_team, away_abbreviation, home_abbreviation,
           spread, predicted_margin
    from serving.srv_game
    where season = 2026
      and season_type = 'regular'
      and home_classification = 'fbs'
      and away_classification = 'fbs'
""", limit=None)

this_week = int(g.loc[~g["is_completed"].astype(bool), "week"].min())
week = g[g["week"] == this_week]
played = int(week["is_completed"].astype(bool).sum())
missing = int((week["spread"].isna() | week["predicted_margin"].isna()).sum())
open_games = week[~week["is_completed"].astype(bool) & week["spread"].notna() & week["predicted_margin"].notna()].copy()

# The gap is computed HERE from today's line, not read from `home_cover_edge`. That column is copied from the
# forecast file at scoring time, so it is frozen at the line the forecast saw (it agreed with the current line
# on only 11 of 56 Week 5 games) and it is empty when the forecast ran before lines existed (all of Week 6).
open_games["gap"] = open_games["spread"] - open_games["predicted_margin"]
open_games = open_games[open_games["gap"].abs() >= MIN_GAP].sort_values("gap")


def line(abbr, value, exact=False):
    """'UGA -24' / 'UGA -17.3': a line from the home team's side, the market's way."""
    text = f"{value:+.2f}".rstrip("0").rstrip(".") if exact else f"{value:+.1f}"
    return f"{abbr} {text}"


def gap_cell(v):
    if abs(v) < 0.05:
        return "0.0"
    bg, fg = ("#dbe9fb", "#0b3d91") if v > 0 else ("#fde3cf", "#8a3b00")
    return f"<span style='background:{bg};color:{fg};font-weight:600;padding:1px 6px'>{v:+.1f}</span>"


table = pd.DataFrame({
    "Kickoff (PT)": pd.to_datetime(open_games["start_date"], utc=True).dt.tz_convert("America/Los_Angeles")
                      .dt.strftime("%a %-m/%-d %-I:%M %p"),
    "Away": open_games["away_team"],
    "Home": open_games["home_team"],
    "Vegas line": [line(a, v, exact=True) for a, v in zip(open_games["home_abbreviation"], open_games["spread"])],
    "My line": [line(a, v) for a, v in zip(open_games["home_abbreviation"], open_games["predicted_margin"])],
    "Gap": [gap_cell(v) for v in open_games["gap"]],
    "You like": ["even" if abs(v) < 0.05 else (h if v > 0 else a)
                 for h, a, v in zip(open_games["home_team"], open_games["away_team"], open_games["gap"])],
    "Matchup": [f"<a href='{SITE}/matchup?season={s}&week={w}&game_id={i}' target='_blank'>open</a>"
                for s, w, i in zip(open_games["season"], open_games["week"], open_games["game_id"])],
})

display(Markdown(f"**Week {this_week}:** {len(table)} open FBS games shown"
                 f" (|gap| ≥ {MIN_GAP:g}) · {played} already played · {missing} missing a line or a forecast"))
display(HTML(table.to_html(index=False, escape=False)))